In [ ]:
from pathlib import Path
import re
from collections.abc import Iterable, Set

import pandas as pd

code_repo = Path().resolve().parent.resolve()
data_repo = (code_repo / "data/shared/sim").resolve()

In [ ]:
code_aux = code_repo / "templates" / "aux"
code_tables_files = code_aux.resolve().glob("life_table*.csv")
data_repo_aux = (data_repo / "aux").resolve().glob("life_table*.csv")
data_repo_scenarios = (data_repo / "scenarios").resolve().glob("*/aux/life_table*.csv")
prefixes: Set[str] = set()
def prefix(p: Path) -> str:
    "Return the unique prefix to use for the columns of each table"
    global prefixes
    t_prefix = table_prefix(p)
    if t_prefix not in prefixes:
        prefixes.add(t_prefix)
        return t_prefix
    t_prefix = f'{location(p)}/{t_prefix}'
    prefixes.add(t_prefix)
    return t_prefix

def table_prefix(p: Path) -> str:
    "Return the prefix to use for the columns of each table, based on filename."
    name = p.stem
    matches = re.match('life_table__(.*)', name)
    if matches is None:
        return 'us'
    return matches[1]

def location(p: Path) -> str:
    p = p.resolve()
    if code_repo.resolve() in p.parents:
        return f'CODE/{p.parent.relative_to(code_repo)}'
    else:
        return f'DATA/{p.parent.relative_to(data_repo)}'

def read_tables(paths: Iterable[Path]) -> dict[str, pd.DataFrame]:
    return {
    p: (pf, pd.read_csv(p)
        .drop(columns="effective_date")
        .rename(columns={"q_male": f'{pf}_male', "q_female": f'{pf}_female'})
        .set_index("age"))
    for p in paths
    for pf in (prefix(p),)
}

tables = read_tables(code_tables_files) | read_tables(data_repo_aux) | read_tables(data_repo_scenarios)
found = pd.Series({key: location(loc) for loc, (key, t) in tables.items()},
                  name="Location").rename_axis("Prefix")
found


In [ ]:

ages = pd.concat((t for (p, t) in tables.values()), axis=1).style.format("{:,.2f}", na_rep='-')
ages
